In [2]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import librosa 
import os
import shutil
import soundfile as sf
import numpy as np
import math


In [3]:

def augment_minority_classes(input_dir, output_dir, target_samples=75):
    """
    Hàm tự động copy dữ liệu gốc và sinh thêm dữ liệu cho lớp thiểu số.
    - input_dir: Thư mục chứa dữ liệu gốc.
    - output_dir: Thư mục đầu ra (an toàn, không ảnh hưởng dữ liệu gốc).
    - target_samples: Mục tiêu số lượng file mỗi lớp (ví dụ: 200).
    """
    # 1. Tạo thư mục đầu ra nếu chưa có
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Lấy danh sách các lớp (các thư mục con)
    classes = [d for d in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, d))]

    for cls in classes:
        cls_input_path = os.path.join(input_dir, cls)
        cls_output_path = os.path.join(output_dir, cls)
        
        if not os.path.exists(cls_output_path):
            os.makedirs(cls_output_path)

        # Lấy danh sách các file audio gốc
        original_files = [f for f in os.listdir(cls_input_path) if f.endswith('.wav')]
        n_current = len(original_files)
        
        print(f"Đang xử lý lớp '{cls}' - Có {n_current} files gốc.")

        # 2. Copy toàn bộ file gốc sang thư mục an toàn
        for f in original_files:
            shutil.copy2(os.path.join(cls_input_path, f), os.path.join(cls_output_path, f))

        # 3. Tính toán xem cần sinh thêm bao nhiêu file
        if n_current < target_samples:
            n_needed = target_samples - n_current
            # Tính số lần cần augment cho mỗi file gốc
            aug_per_file = math.ceil(n_needed / n_current)
            
            files_generated = 0
            
            # Bắt đầu augment
            for f in original_files:
                if files_generated >= n_needed:
                    break # Dừng nếu đã đạt mục tiêu
                    
                file_path = os.path.join(cls_input_path, f)
                y, sr = librosa.load(file_path, sr=None)
                
                for i in range(aug_per_file):
                    if files_generated >= n_needed:
                        break
                        
                    # Lựa chọn ngẫu nhiên kỹ thuật: 0=Noise, 1=Pitch
                    aug_type = np.random.choice([0, 1])
                    y_aug = y.copy()
                    
                    if aug_type == 0:
                        # Thêm Noise
                        noise_amp = 0.005 * np.random.uniform() * np.amax(y_aug)
                        y_aug = y_aug + noise_amp * np.random.normal(size=y_aug.shape[0])
                    else:
                        # Pitch shift (-2 đến 2 bán âm)
                        n_steps = np.random.uniform(-2, 2)
                        y_aug = librosa.effects.pitch_shift(y=y_aug, sr=sr, n_steps=n_steps)
                    
                    # Lưu file mới
                    new_filename = f"aug_{files_generated}_{f}"
                    sf.write(os.path.join(cls_output_path, new_filename), y_aug, sr)
                    files_generated += 1
                    
            print(f" -> Đã tạo thêm {files_generated} files. Tổng: {n_current + files_generated}")
        else:
            print(f" -> Đã đạt chuẩn (>= {target_samples}), không cần augment thêm.")

# --- Cách sử dụng ---
# Gọi hàm để cân bằng các lớp lên mức 200 mẫu

input_dir = r"C:\Users\Hoannd\Downloads\ail\data\corpus\donateacry-organized-wav"

output_dir = r"C:\Users\Hoannd\Downloads\ail\data\corpus_augumented"

target_samples = 100

augment_minority_classes(input_dir,output_dir, target_samples)

Đang xử lý lớp 'belly_pain' - Có 27 files gốc.
 -> Đã tạo thêm 73 files. Tổng: 100
Đang xử lý lớp 'burping' - Có 17 files gốc.
 -> Đã tạo thêm 83 files. Tổng: 100
Đang xử lý lớp 'discomfort' - Có 50 files gốc.
 -> Đã tạo thêm 50 files. Tổng: 100
Đang xử lý lớp 'hungry' - Có 742 files gốc.
 -> Đã đạt chuẩn (>= 100), không cần augment thêm.
Đang xử lý lớp 'lonely' - Có 23 files gốc.
 -> Đã tạo thêm 77 files. Tổng: 100
Đang xử lý lớp 'scared' - Có 15 files gốc.
 -> Đã tạo thêm 85 files. Tổng: 100
Đang xử lý lớp 'tired' - Có 58 files gốc.
 -> Đã tạo thêm 42 files. Tổng: 100
